# Unified codec v2 — reconstruction playground

One model, **any modality** (depth / occupancy / costmap / heatmap / rgb), **any size**, **any rate**.
Shows original / reconstructed / error with PSNR, size, compression, and encode+decode time. Change the
**Config** cell and re-run. The **size sweep** cell demonstrates the size generalization.

In [ ]:
import random, numpy as np, torch, matplotlib.pyplot as plt
from multimodal_ae import MMEncoder, MMDecoder, MAX_DEPTH
import data_multimodal as D

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = MMEncoder().to(DEV).eval(); dec = MMDecoder().to(DEV).eval()
enc.load_state_dict(torch.load('weights/encoder_best_MMV2.pth', map_location=DEV))
dec.load_state_dict(torch.load('weights/decoder_best_MMV2.pth', map_location=DEV))
FILES = D.file_lists()
CMAP = {'depth':'gray','occupancy':'gray','costmap':'gray','heatmap':'inferno'}
print('loaded on', DEV, '| modalities:', list(D.MOD_NCH), '| rates: 4^d for d=1..%d' % MAX_DEPTH)

In [ ]:
def load_one(modality, H, W, idx=None):
    N = D.n_items(modality, FILES)
    if idx is None: idx = random.randrange(N)
    return D._resize(D._load_native(modality, FILES, idx), H, W).numpy(), idx   # (nch,H,W)

def _time(fn, iters=30):
    torch.cuda.synchronize(); s=torch.cuda.Event(True); e=torch.cuda.Event(True); s.record()
    for _ in range(iters): fn()
    e.record(); torch.cuda.synchronize(); return s.elapsed_time(e)/iters

@torch.no_grad()
def infer(x, depth):
    H, W = x.shape[-2:]; xt = torch.from_numpy(x)[None].to(DEV)
    for _ in range(3): dec(enc(xt, depth), depth, H, W)
    enc_ms=_time(lambda: enc(xt, depth)); z=enc(xt, depth); dec_ms=_time(lambda: dec(z, depth, H, W))
    xh = dec(z, depth, H, W).clamp(0,1)[0].cpu().numpy()
    mse = float(((xh-x)**2).mean()); nch=x.shape[0]
    m = dict(enc_ms=enc_ms, dec_ms=dec_ms, mse=mse, psnr=10*np.log10(1/max(mse,1e-9)),
             ratio=(nch*H*W)/z.numel(), orig_mb=nch*H*W*4/1e6, lat_mb=z.numel()*4/1e6, latshape=tuple(z.shape[1:]))
    return xh, m

def show(modality, H, W, depth, idx=None):
    x, idx = load_one(modality, H, W, idx); xh, m = infer(x, depth)
    fig, ax = plt.subplots(1, 3, figsize=(17, 5.2))
    if x.shape[0] == 3:
        o, r = x.transpose(1,2,0), xh.transpose(1,2,0); err = np.abs(r-o).mean(-1)
        ax[0].imshow(o); ax[1].imshow(r)
    else:
        cm = CMAP.get(modality,'gray'); o, r = x[0], xh[0]; err = np.abs(r-o)
        ax[0].imshow(o, cmap=cm, vmin=0, vmax=1); ax[1].imshow(r, cmap=cm, vmin=0, vmax=1)
    ax[0].set_title('original'); ax[1].set_title('reconstructed')
    im = ax[2].imshow(err, cmap='magma', vmin=0, vmax=err.max()+1e-9); ax[2].set_title(f'|error| (max {err.max():.3g})')
    for a in ax: a.axis('off')
    fig.colorbar(im, ax=ax[2], fraction=0.046, pad=0.04)
    fig.suptitle(f"{modality}  {W}x{H}  •  {m['ratio']:.0f}× (depth {depth}, latent {m['latshape']})   |   "
                 f"PSNR {m['psnr']:.1f} dB   |   {m['orig_mb']:.3f}→{m['lat_mb']:.5f} MB   |   "
                 f"enc {m['enc_ms']:.2f} / dec {m['dec_ms']:.2f} ms", fontsize=12)
    plt.tight_layout(); plt.show(); return m

## Config — change these, then run

In [ ]:
MODALITY   = 'rgb'      # depth | occupancy | costmap | heatmap | rgb
H, W       = 512, 768   # ANY size
RATE_DEPTH = 2          # 1..6  ->  4^depth compression

_ = show(MODALITY, H, W, RATE_DEPTH)

## Every modality (same size + rate)

In [ ]:
for mod in D.MOD_NCH:
    show(mod, 512, 768, RATE_DEPTH)

## Size sweep — same content, different sizes (size generalization + latency scaling)

In [ ]:
print(f'{MODALITY} @ {4**RATE_DEPTH}x across sizes:')
for (h, w) in [(128,128),(256,256),(360,640),(512,768),(720,1280),(1080,1920)]:
    x,_ = load_one(MODALITY, h, w, idx=0); _, m = infer(x, RATE_DEPTH)
    print(f'  {w:>4}x{h:<4}  PSNR {m["psnr"]:5.1f} dB   latent {m["lat_mb"]:.4f} MB   enc {m["enc_ms"]:.2f} / dec {m["dec_ms"]:.2f} ms')

## Rate-distortion per modality

In [ ]:
H0, W0, N = 512, 768, 6
plt.figure(figsize=(7.5,5))
colors = plt.cm.viridis(np.linspace(0.1,0.85,len(D.MOD_NCH)))
for mod, c in zip(D.MOD_NCH, colors):
    xs, ys = [], []
    for d in range(1, MAX_DEPTH+1):
        ps=[]
        for k in range(N):
            x,_ = load_one(mod, H0, W0, idx=k); _, m = infer(x, d); ps.append(m['psnr']); lat=m['lat_mb']
        xs.append(lat); ys.append(np.mean(ps))
    plt.plot(xs, ys, '-o', color=c, label=mod, markersize=6, markeredgecolor='black', markeredgewidth=0.6)
plt.xscale('log'); plt.xlabel('Latent size (MB, log)'); plt.ylabel('PSNR (dB)')
plt.title(f'Rate-distortion by modality  ({W0}x{H0}, mean of {N})'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()